### Задание 1
Реализуйте две функции: 
- `span_corruption`. Фунция определяет, какие отрывки предложения заменятся на специальный токен.
- `prepare_pair`. Возвращает обработанные для тренировки пары «вход/выход».

In [19]:
import math
import random
from typing import List, Tuple

random.seed(42)

MASK_RATE = 0.15
MEAN_SPAN = 3.0

def sample_geometric(mean_span: float) -> int:
    """
    Сэмплирование длины спана из геометрического распределения.
    """
    p = 1.0 / mean_span
    span_len = int(math.ceil(math.log(1 - random.random()) / math.log(1 - p)))
    return max(1, span_len)

def span_corruption(tokens: List[str], mask_rate: float = MASK_RATE, mean_span: float = MEAN_SPAN) -> List[Tuple[int, int]]:
    """
    TODO: Верните список спанов для маскирования.
    Каждый спан — это (start, length).
    Маскируем ~mask_rate от общего числа токенов.
    """
    spans = []
    n = len(tokens)
    total_to_mask = max(1, int(round(n * mask_rate)))

    # ===== ВАШ КОД =====
    covered = 0
    i = 0
    # Двигаемся слева направо и сэмплируем спаны, пока не наберется нужная доля
    while covered < total_to_mask and i < n:
        span_len = sample_geometric(mean_span)
        spans.append((i, span_len))
        covered += span_len
        i += span_len + 1  # оставляем зазор
    return spans

def prepare_pair(tokens: List[str], spans: List[Tuple[int, int]]) -> Tuple[str, str]:
    """
    TODO: Построить corrupted_input и target_output.
    Правила:
      - Во входе каждый спан заменяем на <extra_id_k>.
      - В выходе: <extra_id_k> + содержимое спана (все токены).
      - Сентинелы нумеруются слева направо.
    """
    corrupted = []
    target = []
    last_idx = 0
    sentinel_id = 0


    # ===== ВАШ КОД =====
    for start, length in spans:
        # Копируем токены до спана
        span_tokens = tokens[last_idx:start]
        corrupted.extend(span_tokens)
        span_label = f"<extra_id_{sentinel_id}>"
        corrupted.append(span_label)

        # Заполняем target: сентинел + вырезанный спан
        target.append(span_label)
        target.extend(tokens[start:start+length])
        
        last_idx = start + length
        sentinel_id += 1

    # Добавляем хвост после последнего спана
    corrupted.extend(tokens[last_idx:])
    # Добавляем eos
    corrupted.append("<eos>")

    corrupted_text = " ".join(corrupted)
    target_text = " ".join(target)
    return corrupted_text, target_text

# ===== Пример использования =====
tokens = "Модель T5 обучается с помощью span corruption".split()
spans = span_corruption(tokens, mask_rate=0.3)
inp, out = prepare_pair(tokens, spans)

print("Tokens :", tokens)
print("Spans  :", spans)
print("Input  :", inp)
print("Target :", out)

Tokens : ['Модель', 'T5', 'обучается', 'с', 'помощью', 'span', 'corruption']
Spans  : [(0, 3)]
Input  : <extra_id_0> с помощью span corruption <eos>
Target : <extra_id_0> Модель T5 обучается


### Задание 2
Исследуйте датасет C4. Откройте [страницу датасет на huggingface](https://huggingface.co/datasets/allenai/c4/viewer/en/validation) (сплит en/validation). Перейдите в режим DataStudio, справа появится окно для запросов к БД.
С помощью функции `contains(col_name, ‘substring’)` найдите примеры
- новостных данных про:
   - финансы,
   - политику,
   - спорт;
- кулинарных блогов;
- форумов
   - про кино,
   - владельцев BMW.

In [ ]:
-- новости финансов
SELECT *
FROM (SELECT * FROM en_validation LIMIT 100000)
WHERE timestamp BETWEEN '2019-01-01' AND '2020-01-01'
  AND contains("text", 'markets')
LIMIT 10;
-- новости политики
SELECT *
FROM (SELECT * FROM en_validation LIMIT 100000)
WHERE timestamp BETWEEN '2019-01-01' AND '2020-01-01'
  AND contains("text", 'negotiations')
LIMIT 10;
-- новости спорта
SELECT *
FROM (SELECT * FROM en_validation LIMIT 100000)
WHERE timestamp BETWEEN '2019-01-01' AND '2020-01-01'
  AND contains("text", 'gold medal')
LIMIT 10;
-- кулинарный блог
SELECT *
FROM (SELECT * FROM en_validation LIMIT 100000)
WHERE timestamp BETWEEN '2019-01-01' AND '2020-01-01'
  AND contains("text", 'recipe')
LIMIT 10;
-- форум про кино
SELECT *
FROM (SELECT * FROM en_validation LIMIT 100000)
WHERE timestamp BETWEEN '2019-01-01' AND '2020-01-01'
  AND contains("text", 'cinema')
LIMIT 10;
-- форум автовладельцев 
SELECT *
FROM (SELECT * FROM en_validation LIMIT 100000)
WHERE timestamp BETWEEN '2019-01-01' AND '2020-01-01'
  AND contains("text", 'BMW')
LIMIT 10;

### Данные для ruT5
В статье "A Family of Pretrained Transformer Language Models for Russian" была представлена русскоязычная версия модели — ruT5. Для неё использовали те же англоязычные датасеты, их дополнили примерами русскоязычных текстов: датасетом русскоязычных новостей, датасетом русскоязычных книг и срезами из датасета Taiga. 

Модель доступна публично на [странице HuggingFace](https://huggingface.co/ai-forever/ruT5-base).

### Дополнительные материалы
Эти материалы изучать необязательно. Если вам интересно или вы хотите больше погрузиться в тему, то держите ссылки на артефакты, упомянутые в уроке:
- Статья ["Attention is all you need"](https://arxiv.org/abs/1706.03762), в которой был впервые предложен Transformer.
- Статья ["Improving Language Understanding by Generative Pre-Training"](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) — первая GPT от OpenAI.
- Работа ["Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"](https://research.google/blog/exploring-transfer-learning-with-t5-the-text-to-text-transfer-transformer) от Google — возникновение T5.
- Статья ["A Family of Pretrained Transformer Language Models for Russian"](https://arxiv.org/pdf/2309.10931) — о русскоязычной версии модели — ruT5.
- Датасеты для ruT5:
  - [датасет русскоязычных новостей](https://github.com/natasha/corus),
  - [датасет русскоязычных книг](https://huggingface.co/datasets/IlyaGusev/librusec),
  - [срезы из датасета Taiga](https://tatianashavrina.github.io/taiga_site/).
- [T5-Gemma](https://developers.googleblog.com/en/t5gemma/) — страница о современной модели Encoder-Decoder.